In [2]:
ls

'Event extraction evaluation.ipynb'    glm_testiranje.ipynb
'Manual end-to-end evaluation.ipynb'   gpt_model_testing.ipynb


In [3]:
cd ..

/workspace/llm-graph-construction


/opt/conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [5]:
!pip install -q seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=2866ee5531fb943a8e682ce8527336f675ea035a32e624d58c415a0f90c608a8
  Stored in directory: /root/.cache/pip/wheels/1a/67/4a/ad4082dd7dfc30f2abfe4d80a2ed5926a506eb8a972b4767fa
Successfully built seqeval


In [7]:
from training.train_event_extraction import *
from pipeline.pipeline import *

In [8]:
text = "He came to the hospital for a checkup on a mole that appeared two weeks ago."
events = extract_events(text)

In [9]:
events

[(11, 23, 'the hospital'), (28, 37, 'a checkup'), (41, 47, 'a mole')]

# Prepare dataset

In [10]:
from training.train_and_evaluate_relation_extraction import load_stored_dataset_combination_graph
from custom_datasets.dataframe_dataset import DFDataset

In [11]:
_, _, dataset_test_ub = load_stored_dataset_combination_graph(balanced=True, dataset="i2b2")
number_of_relations = 3

In [1]:
dataset_test_ub.df

NameError: name 'dataset_test_ub' is not defined

In [20]:
dataframe = dataset_test_ub.df
documents = set(dataframe["document_id"])
dataset = []
for doc in documents:
    relations = dataframe[dataframe["document_id"] == doc]
    text = relations["text"].iloc[0]
    graph = []
    events = set()
    for row in relations.iloc:
        events.add((row["event1_start"], row["event1_end"], row["event1_text"]))
        events.add((row["event2_start"], row["event2_end"], row["event2_text"]))
        graph.append((row["event1_text"], row["class"], row["event2_text"]))
    dataset.append({"text": text, "graph": graph, "events": events})

# Testing

In [28]:
def evaluate(ground_truth, predictions):
    def overlaps(event1, event2):
        if event1[0]>= event2[0] and event1[0]<=event2[1]:
            return True
        if event1[1]>= event2[0] and event1[1]<=event2[1]:
            return True
        if event2[0]>= event1[0] and event2[0]<=event1[1]:
            return True
        return False

    false_negatives = [True for _ in range(len(ground_truth))]
    true_positives = 0
    false_positives = 0
    for prediction in predictions:
        overlap = False
        for i, gt in enumerate(ground_truth):
            if overlaps(gt, prediction):
                overlap = True
                false_negatives[i] = False
        if overlap:
            true_positives += 1
        else:
            false_positives += 1
    true_negatives = 0
    false_negatives = sum(false_negatives)

    precision = true_positives / (true_positives + false_positives)
    recall = true_positives / len(ground_truth)
    fscore = 2*(precision * recall)/(precision + recall)
    return true_positives, false_positives, len(ground_truth), precision, recall, fscore

In [29]:
all_true_positives = 0
all_false_positives = 0
all_gt_positives = 0
for example in dataset:
    predictions = extract_events(example["text"])
    true_positives, false_positives, gt_positives, precision, recall, fscore = evaluate(example["events"], predictions)
    all_true_positives += true_positives
    all_false_positives += false_positives
    all_gt_positives += gt_positives
# overall scores
precision = all_true_positives / (all_true_positives + all_false_positives)
recall = all_true_positives / all_gt_positives
fscore = 2*(precision * recall)/(precision + recall)
print("precision:", precision)
print("recall:", recall)
print("fscore:", fscore)

precision: 0.8413319551092735
recall: 0.9142329910141207
fscore: 0.8762688403568133
